# The Streaming Data Loop

The full physical-AI data loop in one notebook: **record** a dataset into a
Hugging Face Storage Bucket, **stream** it back with no download, **train** a
policy on the streamed data, and **load** the checkpoint ready to deploy.

This is the notebook version of
[`examples/06_agent_collect_and_stream.py`](https://github.com/strands-labs/robots/blob/main/examples/06_agent_collect_and_stream.py), the agent-driven data-loop example.

**Requirements:** `pip install "strands-robots[sim-mujoco,lerobot]"`
Bucket steps need `hf auth login` + LeRobot >= 0.6.1. Training on GPU needs
`lerobot[training]`. The sim-only path (record + stream from local root) runs
on any laptop, no GPU, no credentials.

In [ ]:
import os
import shutil

# macOS uses "cgl" for offscreen GL; Linux headless uses "egl".
os.environ.setdefault("MUJOCO_GL", "cgl")
os.environ.setdefault("STRANDS_TRUST_REMOTE_CODE", "1")

ROOT = "/tmp/nb5_dataset"
OUT = "/tmp/nb5_ft"
BUCKET = None  # set to "your-org/your-bucket" to enable the bucket path

shutil.rmtree(ROOT, ignore_errors=True)
shutil.rmtree(OUT, ignore_errors=True)

## 1. Record a demonstration

`Robot("so100")` builds a MuJoCo simulation. We add a camera (so the dataset
has video), run the Mock policy for 60 steps (structurally complete data), and
stop recording. The result is a LeRobotDataset on disk.

In [ ]:
from strands_robots import MockPolicy, Robot, create_policy
from strands_robots.training import TrainSpec, create_trainer

sim = Robot("so100", mesh=False)
sim.add_camera(name="front", position=[0.5, 0.0, 0.4], target=[0.2, 0.0, 0.05])
sim.start_recording(
    repo_id="local/nb5_demo",
    root=ROOT,
    fps=30,
    task="pick up the red cube",
    overwrite=True,
)
sim.run_policy(
    robot_name="so100",
    policy_object=MockPolicy(),
    instruction="pick up the red cube",
    n_steps=60,
)
sim.stop_recording()
print("recorded ->", ROOT)

## 2. See what the robot recorded

Render the front camera so you can see the scene the dataset captured.

In [ ]:
from IPython.display import Image, display

# Render the sim before destroying it.
frame = sim.render(camera_name="front")
for item in frame.get("content", []):
    if isinstance(item, dict) and "image" in item:
        display(Image(data=item["image"]["source"]["bytes"], width=480))
        break
sim.destroy()
print("SO-100 arm with front camera - this is what the dataset contains.")

## 3. Sync to a Hugging Face Storage Bucket (optional)

If you have a bucket (`hf buckets create your-org/name --private` after
`hf auth login`), set `BUCKET` in cell 1. The sync uploads only the bytes that
changed (Xet deduplication), so daily re-syncs are fast.

**Skip this cell** if running without credentials (the local path still works
for Steps 4 and 5).

In [ ]:
if BUCKET:
    from strands_robots import Robot as R2

    tmp = R2("so100", mesh=False)
    tmp.stop_recording(bucket=BUCKET)
    tmp.destroy()
    print(f"synced to bucket: hf://buckets/{BUCKET}")
else:
    print("BUCKET not set - skipping sync. Local path works for training below.")

## 4. Stream the dataset back

`stream_dataset()` is the read counterpart to `start_recording()`. It reads
frames lazily with no full download. If you synced to a bucket (Step 3), pass
`repo_type="bucket"` to stream directly from it (requires LeRobot >= 0.6.1).

In [ ]:
sim = Robot("so100", mesh=False)

if BUCKET:
    reader = sim.stream_dataset(BUCKET, repo_type="bucket", shuffle=False)
    print(f"streaming from bucket: {BUCKET}")
else:
    reader = sim.stream_dataset("local/nb5_demo", root=ROOT, shuffle=False)
    print(f"streaming from local root: {ROOT}")

print(f"episodes: {reader.num_episodes} | frames: {reader.num_frames} | fps: {reader.fps}")

for n, frame in enumerate(reader):
    if n == 0:
        img = frame.get("observation.images.front")
        state = frame.get("observation.state")
        print(
            f"frame 0 - image: {tuple(img.shape) if img is not None else None}, state: {tuple(state.shape) if state is not None else None}"
        )
    if n >= 4:
        break
print(f"streamed {n + 1} frames - camera decoded on the fly.")
sim.destroy()

## 5. Train a policy on the dataset

`create_trainer("lerobot_local")` returns a Trainer (the peer of
`create_policy()`). A `TrainSpec` describes the run. Here we train ACT for 2
steps on CPU (fast for the demo). On a GPU with 500 steps this takes about 124
seconds on an NVIDIA L4.

In [ ]:
trainer = create_trainer("lerobot_local", device="cpu")
spec = TrainSpec(
    dataset_root=ROOT,
    base_model="",  # ACT from scratch
    output_dir=OUT,
    steps=2,  # raise to 500+ on a GPU for a real checkpoint
    save_freq=2,
    global_batch_size=2,
    extra={"policy_type": "act", "num_workers": 0},
)

problems = trainer.validate(spec)
assert not problems, problems

result = trainer.train(spec)
print(f"train status: {result.status}")
print(f"checkpoint: {result.checkpoint_dir}")

## 6. Load the trained checkpoint

The same `create_policy()` entry point loads the checkpoint we just produced.
On hardware you would pass `mode="real"` to deploy against a physical arm.

In [ ]:
policy = create_policy(result.checkpoint_dir)
print(f"loaded: {type(policy).__name__}")
print("record -> stream -> train -> load: the data loop closes.")

## Where to go from here

- Raise `steps` to 500 and run on a GPU for a real, deployable checkpoint.
- Set `BUCKET` and run with `hf auth login` to exercise the full bucket path.
- Swap `mode="real"` on `Robot("so100")` to deploy to a physical SO-101.
- Read the [first post in the series](https://huggingface.co/blog/amazon/strands-lerobot-hub-to-hardware) for the full sim-to-hardware walkthrough.
- See [examples/06_agent_collect_and_stream.py](https://github.com/strands-labs/robots/blob/main/examples/06_agent_collect_and_stream.py) for the agent-driven version.